# Lektion 1: Datenbankzugriff mit Python

Übungsnotebook zu Lektion 1 · Hilfsmittel: Skript Lektion 1

In dieser Lektion schreiben Sie den Datenbankzugriff selbst, ohne die Hilfsfunktion `sql()`.
Sie arbeiten mit einer **Kopie** der Datenbank (`uebung_l01.db`). Dort dürfen Sie Daten einfügen und ändern,
ohne die Datenbank für die anderen Lektionen zu verändern. Wenn etwas schiefgeht: Vorbereitungszelle
erneut ausführen, dann ist die Kopie wieder im Ausgangszustand.

> **Lösungsnotebook** – nur für die Lehrkraft

## Vorbereitung

Führen Sie zuerst diese Zelle aus (Umschalt + Enter). Sie legt die Übungskopie der Datenbank an.

In [ ]:
import sqlite3
import shutil
import pandas as pd

# Übungskopie anlegen: hier dürfen Sie alles ändern
shutil.copy("../daten/shop_oltp.db", "uebung_l01.db")
print("Übungskopie uebung_l01.db angelegt")

---
### Aufgabe 1: Verbindung und Cursor

a) Bauen Sie eine Verbindung `con` zur Datei `uebung_l01.db` auf und erzeugen Sie einen Cursor `cur`.

b) Ermitteln Sie mit dem Cursor die Anzahl der Kunden (`SELECT COUNT(*) FROM kunde`) und geben Sie **nur die Zahl** aus, nicht das Tupel.

In [ ]:
con = sqlite3.connect("uebung_l01.db")
cur = con.cursor()

cur.execute("SELECT COUNT(*) FROM kunde")
anzahl = cur.fetchone()[0]
print(anzahl)

**Für die Besprechung:** fetchone() liefert ein Tupel `(40,)`. Mit `[0]` greift man auf das erste Element zu. Häufiger Anfängerfehler: das Tupel direkt weiterverarbeiten.

---
### Aufgabe 2: fetchone, fetchmany, fetchall

Führen Sie `SELECT name FROM kategorie ORDER BY name` aus. Holen Sie dann nacheinander ab:

1. die erste Zeile mit `fetchone()`,
2. die nächsten zwei Zeilen mit `fetchmany(2)`,
3. alle restlichen mit `fetchall()`.

Geben Sie jedes Ergebnis aus. Wie viele Zeilen liefert `fetchall()` und warum nicht alle sechs?

In [ ]:
cur.execute("SELECT name FROM kategorie ORDER BY name")
print(cur.fetchone())
print(cur.fetchmany(2))
print(cur.fetchall())

**Musterlösung:**

fetchall() liefert nur noch 3 Zeilen. Der Cursor hat sich die Position gemerkt: Die ersten drei Zeilen wurden schon abgeholt.

---
### Aufgabe 3: Spaltennamen und Schleife

a) Führen Sie `SELECT * FROM artikel` aus und geben Sie mit `cur.description` eine Liste aller **Spaltennamen** aus.

b) Geben Sie alle Artikel mit einem Listenpreis über 500 € in der Form `Notebook ProBook 14: 899.0 €` aus. Laufen Sie dazu in einer for-Schleife über den Cursor.

In [ ]:
cur.execute("SELECT * FROM artikel")
spalten = [spalte[0] for spalte in cur.description]
print(spalten)

In [ ]:
cur.execute("SELECT bezeichnung, listenpreis FROM artikel WHERE listenpreis > 500")
for bezeichnung, preis in cur:
    print(f"{bezeichnung}: {preis} €")

---
### Aufgabe 4: Eine Funktion mit Platzhalter

Schreiben Sie eine Funktion `kunden_in_ort(ort)`, die Vor- und Nachname aller Kunden eines Ortes als Liste zurückgibt.
Verwenden Sie einen **Platzhalter** für den Ortsnamen.

Testen Sie die Funktion mit `"Köln"` und `"Hamburg"`.

In [ ]:
def kunden_in_ort(ort):
    cur = con.cursor()
    cur.execute("""
        SELECT k.vorname, k.nachname
        FROM kunde k
        JOIN ort o ON o.ort_id = k.ort_id
        WHERE o.name = ?
        ORDER BY k.nachname""", (ort,))
    return cur.fetchall()

print(kunden_in_ort("Köln"))
print(kunden_in_ort("Hamburg"))

**Für die Besprechung:** Auf das Komma in `(ort,)` achten. Ohne Komma ist `(ort)` nur ein geklammerter String, und sqlite3 meldet einen Fehler zur Anzahl der Parameter, weil es jeden Buchstaben als eigenen Wert sieht.

---
### Aufgabe 5: SQL-Injection selbst ausprobieren

Die Funktion unten sucht Kunden nach Nachnamen. Sie baut die Eingabe per f-String in den SQL-Text ein.

a) Rufen Sie sie mit `"Koch"` auf. Wie viele Kunden werden gefunden?
b) Rufen Sie sie mit `"x' OR '1'='1"` auf. Wie viele Kunden werden jetzt gefunden? Lassen Sie sich auch den erzeugten SQL-Text ausgeben.
c) Schreiben Sie eine sichere Funktion `kunde_suchen_sicher(nachname)` mit Platzhalter und testen Sie sie mit beiden Eingaben.

In [ ]:
def kunde_suchen_unsicher(nachname):
    abfrage = f"SELECT vorname, nachname, email FROM kunde WHERE nachname = '{nachname}'"
    print("SQL:", abfrage)
    return con.execute(abfrage).fetchall()

In [ ]:
print(len(kunde_suchen_unsicher("Koch")))
print(len(kunde_suchen_unsicher("x' OR '1'='1")))

In [ ]:
def kunde_suchen_sicher(nachname):
    return con.execute(
        "SELECT vorname, nachname, email FROM kunde WHERE nachname = ?",
        (nachname,)).fetchall()

print(len(kunde_suchen_sicher("Koch")))
print(len(kunde_suchen_sicher("x' OR '1'='1")))

**Musterlösung:**

a) 3 Kunden. b) Alle 40 Kunden mit E-Mail-Adressen. Der erzeugte SQL-Text lautet
`... WHERE nachname = 'x' OR '1'='1'`, die Bedingung ist immer wahr.
c) Mit Platzhalter: 3 Kunden für „Koch“, 0 Kunden für die Angriffseingabe. Sie wird als gewöhnlicher Text gesucht.

**Für die Besprechung:** `con.execute()` ist eine Abkürzung: Sie erzeugt intern einen Cursor und führt den Befehl darauf aus. Gut für die Diskussion: Was wäre bei einer Login-Abfrage mit Benutzername und Passwort möglich?

---
### Aufgabe 6: Daten einfügen: lastrowid und commit

Nordlicht Versand nimmt Drucker ins Sortiment.

a) Legen Sie die Kategorie „Drucker“ an. Geben Sie mit `cur.lastrowid` die neue `kategorie_id` aus.
b) Legen Sie in dieser Kategorie den Artikel „Laserdrucker LX 200“ für 189,00 € an. Verwenden Sie für alle Werte Platzhalter und die ID aus a).
c) Schreiben Sie die Änderungen fest und prüfen Sie mit einer Abfrage, dass der Artikel mit Kategorienamen gespeichert ist.

In [ ]:
cur.execute("INSERT INTO kategorie (name) VALUES (?)", ("Drucker",))
neue_kategorie = cur.lastrowid
print("neue kategorie_id:", neue_kategorie)

cur.execute("INSERT INTO artikel (bezeichnung, kategorie_id, listenpreis) VALUES (?, ?, ?)",
            ("Laserdrucker LX 200", neue_kategorie, 189.00))
con.commit()

cur.execute("""
    SELECT a.artikel_id, a.bezeichnung, k.name, a.listenpreis
    FROM artikel a JOIN kategorie k ON k.kategorie_id = a.kategorie_id
    WHERE k.name = ?""", ("Drucker",))
print(cur.fetchall())

---
### Aufgabe 7: Was passiert ohne commit?

Die Zelle unten öffnet eine **zweite Verbindung** `con2` zur selben Datenbank, so als würde ein Kollege gleichzeitig darauf zugreifen.

1. Fügen Sie über `con` einen neuen Ort ein (PLZ `06108`, Name `Halle (Saale)`, Region 2), aber **ohne** commit.
2. Zählen Sie über `con` und über `con2` die Orte. Was fällt auf?
3. Führen Sie `con.commit()` aus und zählen Sie über `con2` erneut.

In [ ]:
con2 = sqlite3.connect("uebung_l01.db")
print("zweite Verbindung geöffnet")

In [ ]:
con.execute("INSERT INTO ort (plz, name, region_id) VALUES (?, ?, ?)", ("06108", "Halle (Saale)", 2))

print("con  sieht:", con.execute("SELECT COUNT(*) FROM ort").fetchone()[0])
print("con2 sieht:", con2.execute("SELECT COUNT(*) FROM ort").fetchone()[0])

con.commit()
print("nach commit, con2 sieht:", con2.execute("SELECT COUNT(*) FROM ort").fetchone()[0])
con2.close()

**Musterlösung:**

Vor dem commit sieht `con` 13 Orte, `con2` nur 12. Die Änderung ist vorläufig und nur innerhalb der eigenen Transaktion sichtbar.
Nach dem commit sieht auch `con2` 13 Orte. Wäre `con` ohne commit geschlossen worden, wäre der neue Ort verloren.

**Für die Besprechung:** Schöne Vorbereitung auf das I in ACID (Isolation) in Lektion 10.

---
### Aufgabe 8: Transaktion mit rollback

Kundin 5 bestellt per Telefon zwei Artikel. Die Bestellung soll **ganz oder gar nicht** gespeichert werden.

a) Speichern Sie Bestellung 321 (Kunde 5, Mitarbeiter 3, Bestelldatum 2025-12-15, kein Lieferdatum, Status „offen“) mit zwei Positionen:
Position 1: Artikel 1, Menge 1, Einzelpreis 899.00 · Position 2: Artikel 4, Menge 2, Einzelpreis 159.00.
Bauen Sie absichtlich einen Fehler ein: Geben Sie beiden Positionen die Positionsnummer 1.
Fangen Sie den Fehler mit `try` / `except sqlite3.IntegrityError` ab und rufen Sie dann `rollback()` auf.

b) Prüfen Sie: Gibt es Bestellung 321 in der Datenbank?

c) Korrigieren Sie den Fehler und speichern Sie die Bestellung erfolgreich.

In [ ]:
def bestellung_speichern(positionsnummern):
    try:
        cur.execute("INSERT INTO bestellung VALUES (?, ?, ?, ?, ?, ?)",
                    (321, 5, 3, "2025-12-15", None, "offen"))
        cur.execute("INSERT INTO bestellposition VALUES (?, ?, ?, ?, ?, ?)",
                    (321, positionsnummern[0], 1, 1, 899.00, None))
        cur.execute("INSERT INTO bestellposition VALUES (?, ?, ?, ?, ?, ?)",
                    (321, positionsnummern[1], 4, 2, 159.00, None))
        con.commit()
        print("Bestellung gespeichert")
    except sqlite3.IntegrityError as fehler:
        con.rollback()
        print("Bestellung nicht gespeichert:", fehler)

bestellung_speichern((1, 1))   # a) mit Fehler

In [ ]:
# b)
cur.execute("SELECT COUNT(*) FROM bestellung WHERE bestellung_id = ?", (321,))
print(cur.fetchone()[0])

In [ ]:
# c)
bestellung_speichern((1, 2))
cur.execute("SELECT * FROM bestellposition WHERE bestellung_id = ?", (321,))
print(cur.fetchall())

**Musterlösung:**

b) 0. Der rollback hat auch den bereits eingefügten Bestellkopf und die erste Position zurückgenommen.

**Für die Besprechung:** Die Funktion ist eine elegante, aber nicht verlangte Lösung. Drei einzelne Zellen sind genauso richtig.

---
### Aufgabe 9: Viele Datensätze mit executemany

Der Vertrieb liefert eine Liste neuer Liefergebiete. Fügen Sie alle mit **einem** Aufruf von `executemany()` in die Tabelle `ort` ein, schreiben Sie fest und geben Sie `cur.rowcount` aus.
Wie viele Orte gibt es danach insgesamt?

In [ ]:
neue_orte = [
    ("99084", "Erfurt", 2),
    ("66111", "Saarbrücken", 4),
    ("30159", "Hannover", 1),
    ("60311", "Frankfurt am Main", 4),
    ("93047", "Regensburg", 3),
]

In [ ]:
cur.executemany("INSERT INTO ort (plz, name, region_id) VALUES (?, ?, ?)", neue_orte)
con.commit()
print("eingefügt:", cur.rowcount)

cur.execute("SELECT COUNT(*) FROM ort")
print("Orte insgesamt:", cur.fetchone()[0])

**Musterlösung:**

5 eingefügt, insgesamt 18 Orte (12 ursprüngliche, Halle aus Aufgabe 7 und die 5 neuen).

**Für die Besprechung:** Wurde Aufgabe 7 übersprungen, sind es 17. Der Befehl wird einmal vorbereitet und fünfmal mit anderen Werten ausgeführt.

---
### Aufgabe 10: pandas mit Parametern

a) Ermitteln Sie mit `pd.read_sql_query()` den Umsatz je Jahr (`strftime('%Y', bestelldatum)`) für **Kunde 23**, ohne stornierte Bestellungen. Übergeben Sie die Kunden-ID als Parameter.

b) In den anderen Notebooks gibt es die Hilfsfunktion `sql()`. Erklären Sie in zwei bis drei Sätzen, was sie tut.

In [ ]:
pd.read_sql_query("""
    SELECT strftime('%Y', b.bestelldatum) AS jahr,
           SUM(p.menge * p.einzelpreis) AS umsatz
    FROM bestellung b
    JOIN bestellposition p ON p.bestellung_id = b.bestellung_id
    WHERE b.kunde_id = ? AND b.status <> 'storniert'
    GROUP BY jahr""", con, params=(23,))

**Musterlösung:**

b) `sql()` übergibt den SQL-Text zusammen mit der Connection `con` an `pd.read_sql_query()` und gibt das Ergebnis als DataFrame zurück. Tritt ein Fehler auf, wird er abgefangen und nur die Ursache ausgegeben. Parameter unterstützt sie nicht, sie ist nur für Übungsabfragen ohne Eingaben von außen gedacht.

---
### Aufgabe 11: Prüfungsaufgabe: Code beurteilen

In einer Webanwendung finden Sie folgenden Java-Code für die Anmeldung:

```java
String sql = "SELECT * FROM benutzer WHERE login = '" + login
           + "' AND passwort_hash = '" + hash + "'";
ResultSet rs = statement.executeQuery(sql);
```

a) Benennen Sie die Sicherheitslücke und beschreiben Sie einen möglichen Angriff.
b) Wie heißt die wirksamste Gegenmaßnahme? Schreiben Sie die ersten beiden Zeilen entsprechend um.
c) Ein Kollege schlägt vor, auch den Tabellennamen über einen Platzhalter zu übergeben: `SELECT * FROM ? WHERE ...`. Ist das möglich?

Schließen Sie zum Abschluss Ihre Verbindung mit `con.close()`.

In [ ]:
con.close()

**Musterlösung:**

a) SQL-Injection. Die Eingabe `login` wird per Verkettung in den SQL-Text eingebaut. Gibt ein Angreifer als Login
`admin' --` ein, wird der Rest der Bedingung zum Kommentar. Die Passwortprüfung entfällt, er ist als admin angemeldet.

b) Prepared Statement (parametrisierte Abfrage):

```java
PreparedStatement ps = con.prepareStatement(
    "SELECT * FROM benutzer WHERE login = ? AND passwort_hash = ?");
ps.setString(1, login);
ps.setString(2, hash);
ResultSet rs = ps.executeQuery();
```

c) Nein. Platzhalter stehen nur für Werte, nicht für Tabellen- oder Spaltennamen, denn diese müssen beim Vorbereiten
schon feststehen. Wenn ein Tabellenname variabel sein muss, prüft man ihn gegen eine feste Liste erlaubter Namen.